# 🧬 UltraMuscleAI: Exploratory Data Analysis & Preprocessing Pipeline

This notebook demonstrates and validates the data preprocessing steps for the **UMUD Challenge**:
1. **Dataset Discovery & Audit**
2. **Deduplication Audit (MD5 & Perceptual Hashing)**
3. **Probe Depth & Scale Metrology (px-to-mm)**
4. **Leak-Free Stratified Group K-Fold Cross-Validation**
5. **Ultrasound Speckle Augmentations & Angle-Preserving Flips**
6. **PyTorch DataLoader Verification**

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

# Set project root
PROJECT_ROOT = Path("D:/UltraMuscleAI")
sys.path.append(str(PROJECT_ROOT))

from src.data.transforms import get_ultrasound_transforms, apply_angle_preserving_horizontal_flip
from src.data.dataset import UMUDSegmentationDataset, UMUDRegressionDataset
from src.metrology.scale_detector import UltrasoundScaleDetector
from src.data.preprocess import run_preprocessing_pipeline

print("Project modules imported successfully.")

ModuleNotFoundError: No module named 'torch'

## 1. Execute Preprocessing & Deduplication Pipeline

In [2]:
# Run automated preprocessing, deduplication audit, scale calibration, and 5-fold split
results = run_preprocessing_pipeline(
    raw_dir=PROJECT_ROOT / "data" / "raw",
    processed_dir=PROJECT_ROOT / "data" / "processed",
    n_splits=5,
    auto_generate_sample=True
)
print("Pipeline summary:", results)

NameError: name 'run_preprocessing_pipeline' is not defined

## 2. Inspect Cleaned Folds Manifest

In [ ]:
folds_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "folds.csv")
display(folds_df.head(10))
print(f"Total clean samples: {len(folds_df)}")
print("Fold counts:\n", folds_df['fold'].value_counts().sort_index())

## 3. Visualize Architectural Parameter Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(folds_df['pa_deg'], bins=15, color='#20BEFF', edgecolor='black')
axes[0].set_title('Pennation Angle (θ_PA, deg)')
axes[0].set_xlabel('Degrees')

axes[1].hist(folds_df['fl_mm'], bins=15, color='#4CAF50', edgecolor='black')
axes[1].set_title('Fascicle Length (FL, mm)')
axes[1].set_xlabel('mm')

axes[2].hist(folds_df['mt_mm'], bins=15, color='#FF5722', edgecolor='black')
axes[2].set_title('Muscle Thickness (MT, mm)')
axes[2].set_xlabel('mm')

plt.tight_layout()
plt.show()

## 4. Test Ultrasound Augmentations & Angle Inversion

In [ ]:
sample_row = folds_df.iloc[0]
img = cv2.imread(sample_row['image_path'])
mask = cv2.imread(sample_row['mask_path'], cv2.IMREAD_GRAYSCALE)

# Angle-preserving horizontal flip test
flipped_img, flipped_mask, flipped_targets = apply_angle_preserving_horizontal_flip(
    image=img,
    mask=mask,
    pa_deg=sample_row['pa_deg'],
    fl_mm=sample_row['fl_mm'],
    mt_mm=sample_row['mt_mm']
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original (PA: {sample_row['pa_deg']}°)")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(flipped_img, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Flipped (PA: {flipped_targets['pa_deg']}°)")
axes[1].axis('off')

plt.show()